# PHASE 2 - Feature Engineering

This notebook creates calendar, lag, and rolling features from the hourly bike-demand data. It does not train a model or split the data.

Leakage rule: every lag and rolling feature is calculated from earlier rows only. In particular, rolling features use `target.shift(1)` so the target at the prediction timestamp is never included.

In [5]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

data_candidates = [
    Path("data/raw/hour.csv"),
    Path("../data/raw/hour.csv"),
]
data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Place the dataset at data/raw/hour.csv before running this notebook.")

df = pd.read_csv(data_path)
df["dteday"] = pd.to_datetime(df["dteday"], errors="raise")
df["timestamp"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded: {data_path}")
print(f"Raw shape: {df.shape}")
print(f"Timestamp range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

Loaded: ..\data\raw\hour.csv
Raw shape: (17379, 18)
Timestamp range: 2011-01-01 00:00:00 to 2012-12-31 23:00:00


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,timestamp
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2011-01-01 00:00:00
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2011-01-01 01:00:00
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2011-01-01 02:00:00
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,2011-01-01 03:00:00
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,2011-01-01 04:00:00


## Calendar features

In [6]:
df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month
df["year"] = df["timestamp"].dt.year
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["day_of_year"] = df["timestamp"].dt.dayofyear
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["is_workingday"] = df["workingday"].astype(int)
df["rush_hour"] = df["hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

calendar_features = [
    "timestamp", "hour", "day", "month", "year",
    "day_of_week", "day_of_year", "is_weekend",
    "is_workingday", "season", "rush_hour",
]
df[calendar_features].head()

,timestamp,hour,day,month,year,day_of_week,day_of_year,is_weekend,is_workingday,season,rush_hour
0,2011-01-01 00:00:00,0,1,1,2011,5,1,1,0,1,0
1,2011-01-01 01:00:00,1,1,1,2011,5,1,1,0,1,0
2,2011-01-01 02:00:00,2,1,1,2011,5,1,1,0,1,0
3,2011-01-01 03:00:00,3,1,1,2011,5,1,1,0,1,0
4,2011-01-01 04:00:00,4,1,1,2011,5,1,1,0,1,0


## Past-demand lag and rolling features

The dataset is ordered chronologically before `shift`. Therefore `lag_1`, `lag_2`, `lag_24`, and `lag_168` refer to earlier observations in the time-ordered data. The rolling windows are shifted first, which excludes the current target.

In [ ]:
target = df["cnt"]

for lag in [1, 2, 24, 168]:
    df[f"lag_{lag}"] = target.shift(lag)

past_target = target.shift(1)
df["rolling_mean_24"] = past_target.rolling(window=24, min_periods=24).mean()
df["rolling_mean_168"] = past_target.rolling(window=168, min_periods=168).mean()

lag_features = ["lag_1", "lag_2", "lag_24", "lag_168"]
rolling_features = ["rolling_mean_24", "rolling_mean_168"]
weather_features = ["temp", "atemp", "hum", "windspeed", "weathersit"]
feature_columns = calendar_features[1:] + weather_features + lag_features + rolling_features

df[["timestamp", "cnt"] + lag_features + rolling_features].head(170)

# Verify that rolling features use only targets from earlier rows.
rolling_start = int(df["rolling_mean_24"].first_valid_index())
previous_targets = df["cnt"].iloc[rolling_start - 24:rolling_start]
expected_rolling = previous_targets.mean()
actual_rolling = df["rolling_mean_24"].iloc[rolling_start]
assert abs(expected_rolling - actual_rolling) < 1e-9
print("Leakage check passed: rolling_mean_24 uses the 24 preceding targets.")

before_drop = len(df)
df_features = df.dropna(subset=lag_features + rolling_features).reset_index(drop=True)
print(f"Rows before removing insufficient history: {before_drop}")
print(f"Rows after feature engineering: {len(df_features)}")
print("Remaining missing values in model features:", int(df_features[feature_columns].isna().sum().sum()))
df_features[["timestamp", "cnt"] + feature_columns].head()

project_root = Path(".") if Path("data/raw/hour.csv").exists() else Path("..")
processed_dir = project_root / "data/processed"
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "hour_features.csv"
df_features.to_csv(output_path, index=False)
print(f"Saved processed features to: {output_path}")
print(f"Feature count: {len(feature_columns)}")

Leakage check passed: rolling_mean_24 uses the 24 preceding targets.
Rows before removing insufficient history: 17379
Rows after feature engineering: 17211
Remaining missing values in model features: 0
Saved processed features to: data\processed\hour_features.csv
Feature count: 21
